# Lab 3 — Detección de Anomalías en Tráfico de Red con Machine Learning

**Curso:** Seguridad Informática | **Unidad IV:** Monitoreo, SIEM e IA  
**Alumno:** *(Tu nombre aquí)*  
**Dataset:** `lab3/network_traffic.csv` — 10 000 registros de tráfico de 30 días

In [ ]:
# Instalación de dependencias (ejecutar una sola vez)
import subprocess, sys
pkgs = ['pandas','numpy','matplotlib','seaborn','scikit-learn','joblib']
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p])
print('Dependencias listas.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay)
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Librerías importadas OK')

---
## Tarea 3.1 — Exploración y Preprocesamiento

In [ ]:
# Cargar dataset
df = pd.read_csv('lab3/network_traffic.csv', parse_dates=['timestamp'])
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Estadísticas descriptivas
df.describe()

In [ ]:
# Tipos y valores nulos
print('Tipos de datos:')
print(df.dtypes)
print('\nValores nulos por columna:')
print(df.isnull().sum())

In [ ]:
# Distribución de bytes_sent y duration_sec
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['bytes_sent'], bins=50, color='steelblue', edgecolor='white', log=True)
axes[0].set_title('Distribución de bytes_sent (escala log)', fontsize=13)
axes[0].set_xlabel('Bytes enviados')
axes[0].set_ylabel('Frecuencia (log)')

axes[1].hist(df['duration_sec'], bins=50, color='darkorange', edgecolor='white', log=True)
axes[1].set_title('Distribución de duration_sec (escala log)', fontsize=13)
axes[1].set_xlabel('Duración (segundos)')
axes[1].set_ylabel('Frecuencia (log)')

plt.tight_layout()
plt.savefig('lab3/evidencias/SCR-3.1_eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('Histogramas guardados.')

In [ ]:
# Tratar valores nulos y atípicos extremos
numericas = ['bytes_sent', 'bytes_recv', 'duration_sec', 'packets', 'dst_port']
for col in numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(subset=numericas, inplace=True)

# Clip extremos (percentil 99.9) para reducir ruido sin eliminar anomalías
for col in ['bytes_sent', 'bytes_recv']:
    upper = df[col].quantile(0.999)
    df[col] = df[col].clip(upper=upper)

print(f'Registros tras limpieza: {len(df)}')

In [ ]:
# Feature engineering: 2 nuevas variables derivadas
df['ratio_bytes']      = df['bytes_sent'] / (df['bytes_recv'] + 1)   # relación sent/recv
df['bytes_por_segundo'] = (df['bytes_sent'] + df['bytes_recv']) / (df['duration_sec'] + 1)  # throughput

print('Nuevas features creadas:')
print(df[['ratio_bytes', 'bytes_por_segundo']].describe())

In [ ]:
# Normalización con StandardScaler
FEATURES = ['dst_port', 'bytes_sent', 'bytes_recv', 'duration_sec',
            'packets', 'ratio_bytes', 'bytes_por_segundo']

X = df[FEATURES].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Normalización completada.')
print(f'Shape X_scaled: {X_scaled.shape}')

# Guardar scaler para predecir.py
import os; os.makedirs('lab3', exist_ok=True)
joblib.dump(scaler, 'lab3/scaler.pkl')
print('scaler.pkl guardado.')

---
## Tarea 3.2 — Entrenamiento del Modelo Isolation Forest

In [ ]:
# Entrenar Isolation Forest (sin usar la columna 'label')
modelo = IsolationForest(
    contamination=0.05,
    n_estimators=100,
    random_state=42
)
modelo.fit(X_scaled)
print('Modelo entrenado.')

In [ ]:
# Predicciones
pred_raw = modelo.predict(X_scaled)   # -1 anomalía, 1 normal
scores   = modelo.decision_function(X_scaled)

# Convertir a etiquetas binarias para métricas (1 = anomalía, 0 = normal)
y_pred = (pred_raw == -1).astype(int)
y_true = (df['label'] == 'anomaly').astype(int)

precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)

print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1-Score  : {f1:.4f}')

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Anomalía'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matriz de Confusión — Isolation Forest', fontsize=13)
plt.tight_layout()
os.makedirs('lab3/evidencias', exist_ok=True)
plt.savefig('lab3/evidencias/SCR-3.2_metricas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Matriz guardada.')

---
## Tarea 3.3 — Interpretación y Umbral Dinámico

In [ ]:
# Gráfico: anomaly score de todos los registros
fig, ax = plt.subplots(figsize=(14, 4))
ax.scatter(range(len(scores)), scores, c=y_true, cmap='RdYlGn',
           alpha=0.4, s=5, label='Score')
ax.axhline(0, color='red', linestyle='--', linewidth=1.5, label='Umbral default (0)')
ax.set_xlabel('Índice de registro')
ax.set_ylabel('Anomaly Score (decision_function)')
ax.set_title('Anomaly Score por registro (verde=normal, rojo=anomalía real)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Curva umbral vs F1-Score
umbrales = np.linspace(scores.min(), scores.max(), 200)
f1_scores = []

for u in umbrales:
    y_u = (scores < u).astype(int)
    f1_scores.append(f1_score(y_true, y_u, zero_division=0))

umbral_optimo = umbrales[np.argmax(f1_scores)]
f1_optimo = max(f1_scores)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(umbrales, f1_scores, color='royalblue', linewidth=2)
ax.axvline(umbral_optimo, color='red', linestyle='--',
           label=f'Umbral óptimo = {umbral_optimo:.4f} (F1={f1_optimo:.4f})')
ax.set_xlabel('Umbral (decision_function)')
ax.set_ylabel('F1-Score')
ax.set_title('Curva Umbral vs F1-Score', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('lab3/evidencias/SCR-3.3_umbral_f1.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Umbral óptimo: {umbral_optimo:.4f}')
print(f'F1 máximo   : {f1_optimo:.4f}')

In [ ]:
# Top 10 registros más anómalos
df['anomaly_score'] = scores
top10_anomalias = df.nsmallest(10, 'anomaly_score')
top10_anomalias[['timestamp','src_ip','dst_ip','dst_port','protocol',
                  'bytes_sent','bytes_recv','duration_sec','packets',
                  'label','anomaly_score']]

### Interpretación — Top 10 Registros Más Anómalos

Los 10 registros con score más bajo (más negativos) representan las conexiones más alejadas
del comportamiento normal del tráfico. Las características que los hacen sospechosos son:

1. **Volumen extremo de bytes_sent o bytes_recv**: transferencias de datos muy por encima
   de la media sugieren exfiltración de datos o descarga masiva no autorizada.
2. **ratio_bytes muy alto**: si `bytes_sent >> bytes_recv`, la máquina interna está enviando
   mucho más de lo que recibe, patrón típico de exfiltración o C&C beaconing.
3. **Puertos inusuales con alto throughput**: conexiones a puertos no estándar (ej. 4444, 1337)
   con gran volumen pueden indicar backdoors o túneles encubiertos.
4. **Duración anómala**: conexiones muy largas (hours) o muy cortas con mucho tráfico
   pueden indicar DDoS, escaneos o transferencias automatizadas maliciosas.
5. **Protocolo ICMP con alto bytes_sent**: normalmente ICMP lleva payloads pequeños;
   un ICMP con megabytes transferidos es un indicador claro de ICMP tunneling.

---
## Tarea 3.4 — Exportación del Modelo

In [ ]:
# Serializar modelo con joblib
joblib.dump(modelo, 'lab3/modelo_anomalias.pkl')
print('Modelo guardado en: lab3/modelo_anomalias.pkl')
print('Scaler guardado en: lab3/scaler.pkl')
print('\nPara predecir en nuevos datos:')
print('  python predecir.py nuevo_trafico.csv')